In [ ]:
import os
import pandas as pd
from omegaconf import OmegaConf

from hate_meme.data_processing.annotate import *

In [ ]:
cfg = OmegaConf.load("../../config/default.yaml")

all_hateful_data = os.path.expanduser(cfg.data.paths.all_hateful)
hf_data = os.path.expanduser(cfg.data.paths.hf)
kaggle_data = os.path.expanduser(cfg.data.paths.kaggle)
labels_dir = os.path.expanduser(cfg.out.hateful_memes_labels)

In [ ]:
df = pd.read_csv(f'{all_hateful_data}/hateful_memes_all.csv')

In [ ]:
# Add columns to indicate where the image is found
def add_img_path_prefix(row):
    if row['img_found_hf']:
        return os.path.join(hf_data, row['img'])
    elif row['img_found_kaggle']:
        return os.path.join(kaggle_data, row['img'])
    else:
        return None

df['img'] = df.apply(add_img_path_prefix, axis=1)

# Drop rows where the image is not found
print(f"Number of dropped rows: {len(df[df['img'].isnull()])}")
df = df[df['img'].notnull()]

# Drop duplicate images
print(f"Number of dropped duplicated images: {df['img'].duplicated().sum()}")
df = df.drop_duplicates(subset=['img'], keep='first')

# Drop irrelevant images
relevant_imgs = []
with open('matched_img_ids.txt', 'r') as f:
    for line in f:
        # Remove unwanted characters and extract the number
        filename = line.strip().split('/')[-1].split('.')[0]
        relevant_imgs.append(int(filename))
print(f"Number of irrelevant images: {len(df) - len(relevant_imgs)}")
df_relevant = df[df['id'].isin(relevant_imgs)]

In [ ]:
# Create a new 'split' column with simplified values
split_map = {
    'train': 'train',
    'test_seen': 'test',
    'test_unseen': 'test',
    'dev_seen': 'dev',
    'dev_unseen': 'dev'
}

df_relevant.loc[:, 'split'] = df_relevant['source_hf'].map(split_map)

In [ ]:
# Sample 20% of the DataFrame, keeping split proportions, with a fixed seed
sampled_df = df_relevant.groupby('split').sample(frac=0.2, random_state=42, replace=False)

In [ ]:
annotate_data(sampled_df, f'{labels_dir}/labels.jsonl', 'Nils')

### Stats on Annotation

In [ ]:
import json

# Load the JSONL file into a DataFrame
with open(f'{labels_dir}/labels.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]

labels_df = pd.DataFrame(data)

In [ ]:
# Create binary labels
labels_df['incivility'] = labels_df['label_incivility'].apply(lambda x: 1 if x else 0)
labels_df['intolerance'] = labels_df['label_intolerance'].apply(lambda x: 1 if x else 0)

In [ ]:
for split in [df, sampled_df]:
    merged_df = split.merge(labels_df[['id', 'incivility', 'intolerance']], on='id', how='inner')
    print("Nr. labels:", len(merged_df))
    # Overlap with hateful memes labels
    print("P(hateful):", end=' ')
    print(sum(merged_df['label_hf'] == 1) / len(merged_df))

    print("P(hateful = incivil)", end=' ')
    print(sum(merged_df['incivility'] == merged_df['label_hf']) / len(merged_df))

    print("P(hateful = intolerant)", end=' ')
    print(sum(merged_df['intolerance'] == merged_df['label_hf']) / len(merged_df))

    print("P(hateful = intolerant AND uncivil)", end=' ')
    print(sum((merged_df['incivility'] & merged_df['intolerance']) == merged_df['label_hf']) / len(merged_df))

    print("P(hateful = intolerant OR uncivil)", end=' ')
    print(sum((merged_df['incivility'] | merged_df['intolerance']) == merged_df['label_hf']) / len(merged_df))
    print()

In [ ]:
not_match = merged_df[merged_df['intolerance'] != merged_df['label_hf']]
for i in range(len(not_match)):
    row = not_match.iloc[i]
    img_path = row['img']
    clear_output(wait=True)
    try:
        display_fixed_image(img_path, size=(512, 512))
    except FileNotFoundError:
        print(f"Image not found: {img_path}")
    
    user_input = input()
    if user_input == '':
        continue
    elif user_input == 'q':
        break